In [108]:
! pip install scikit-optimize

In [109]:
import numpy as np
import os
import time
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from skopt import BayesSearchCV
from skopt.space import Real, Categorical

np.random.seed(1)

In [110]:
def load_data(folder_path):
    x_train = np.load(os.path.join(folder_path, 'x_train.npy'))
    y_train = np.load(os.path.join(folder_path, 'y_train.npy'))
    x_test = np.load(os.path.join(folder_path, 'x_test.npy'))
    y_test = np.load(os.path.join(folder_path, 'y_test.npy'))
    return x_train, y_train, x_test, y_test

In [111]:
x_train, y_train, x_test, y_test = load_data('lr4_dataset/')

В данной лабораторной работе будет практиковаться поиск гиперпараметров. Буду рассмотрены алгоритмы поиска гиперпараметров: grid search, random search.

Помимо поиска гиперпараметров будет рассмотрен алгоритм кросс-валидации, позволяющий получить более достоверную оценку качества модели в условиях недостатка данных.
Хотя в работе предоставлена тестовая выборка, здесь она имеет сугубо теоретический характер (для получения финальной оценки) и на практике как правило недоступна. Поэтому во время подбора гиперпараметров используются лишь `x_train, y_train`. `x_test, y_test` используются лишь для получения финальной оценки, чтобы можно было видеть разницу между разными алгоритмами подбора гиперпараметров (если она будет).

Выберите одну модель из списка: MLPClassifier, SGDClassifier, DecisionTreeClassifier, RandomForestClassifier, SVC.
Для выбранной модели произведите поиск оптимальных гиперпараметров.

**Требование**: поиск должен идти как минимум для двух гиперпараметров.

**Требование**: в конструктор моделей передавайте `random_state=1` для воспроизводимости результатов.

## 0. Обучение бейзлайн модели для проведения сравнения

In [112]:
# Обучите базовую модель без изменения гиперпараметров
# (т.е. используются гиперпараметры по умолчанию).
# Проанализируйте качество модели (accuracy, матрица ошибок).

print("Базовая модель")
base_model = SGDClassifier(random_state=1)

print("Параметры модели по умолчанию:")
for param, value in base_model.get_params().items():
    print(f"  {param}: {value}")

base_model.fit(x_train, y_train)
y_pred_base = base_model.predict(x_test)
base_accuracy = accuracy_score(y_test, y_pred_base)

print(f"\nAccuracy: {base_accuracy:.4f}")
print("\nМатрица ошибок:")
print(confusion_matrix(y_test, y_pred_base))

Базовая модель
Параметры модели по умолчанию:
  alpha: 0.0001
  average: False
  class_weight: None
  early_stopping: False
  epsilon: 0.1
  eta0: 0.0
  fit_intercept: True
  l1_ratio: 0.15
  learning_rate: optimal
  loss: hinge
  max_iter: 1000
  n_iter_no_change: 5
  n_jobs: None
  penalty: l2
  power_t: 0.5
  random_state: 1
  shuffle: True
  tol: 0.001
  validation_fraction: 0.1
  verbose: 0
  warm_start: False

Accuracy: 0.6000

Матрица ошибок:
[[3 0 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0]
 [0 0 1 2 0 0 0 0 0 0]
 [0 0 0 0 2 1 0 0 0 0]
 [0 0 0 0 0 3 0 0 0 0]
 [1 0 0 0 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 3 0 0]
 [0 0 1 0 0 1 0 0 1 0]
 [0 0 1 0 0 0 0 2 0 0]]


## 1. K-Fold Cross-Validation

In [113]:
# Реализуйте функцию кросс-валидации.
# Замечание: x_test, y_test не должны применяться внутри этой функции.

def kfold_cv(model_fn, eval_fn, x: np.ndarray, y: np.ndarray,
             n_splits: int = 5) -> float:
    """
    Реализует K-fold кросс-валидацию.

    Parameters
    ----------
    model_fn : callable
        Функция-фабрика, которая конструирует и возвращает новый объект
        модели.
    eval_fn : callable
        Функция вида eval_fn(labels, predictions), которая возвращает скалярное
        значение метрики.
    x : np.ndarray
        Набор признаков (размерность NxD, где N - количество экземпляров,
        D - количество признаков).
    y : np.ndarray
        Набор меток (размерность N).
    n_splits : int, optional
        Количество фолдов (подвыборок). По умолчанию 5.

    Returns
    -------
    float
        Среднее значение метрики (вычисляемой eval_fn) по всем фолдам.
    """
    indices = np.arange(len(x))
    np.random.shuffle(indices)

    fold_size = len(x) // n_splits
    scores = []

    for i in range(n_splits):
        start_idx = i * fold_size
        end_idx = (i + 1) * fold_size if i < n_splits - 1 else len(x)

        val_indices = indices[start_idx:end_idx]
        train_indices = np.concatenate([indices[:start_idx],
                                        indices[end_idx:]])

        x_train_fold, x_val_fold = x[train_indices], x[val_indices]
        y_train_fold, y_val_fold = y[train_indices], y[val_indices]

        model = model_fn()
        model.fit(x_train_fold, y_train_fold)
        y_pred = model.predict(x_val_fold)

        score = eval_fn(y_val_fold, y_pred)
        scores.append(score)

    return np.mean(scores)

In [114]:
# Убедитесь в корректности работы функции кросс-валидации.

print("Проверка кросс-валидации")

def create_base_model():
    """
    Фабричная функция для создания базовой модели SGDClassifier.
    """
    return SGDClassifier(random_state=1)

check_score = kfold_cv(create_base_model, accuracy_score,
                       x_train, y_train)
print(f"CV Score: {check_score:.4f}")

Проверка кросс-валидации
CV Score: 0.5182


## 2. Grid search

In [115]:
# 1. Реализуйте алгоритм поиска гиперпараметров grid search.
# 2. Запустите поиск гиперпараметров, замерьте время работы
# алгоритма.
# 3. Выведите найденные значения гиперпараметров и время работы.
# Замечание: x_test, y_test не должны применяться в рамках
# данного алгоритма.
# Замечание: убедитесь, что гиперпараметры по умолчанию включены
# в пространство поиска.
# Требование: используйте kfold_cv для получения значения метрики
# в рамках одной итерации поиска гиперпараметров.

print("Grid Search")

param_grid = {
    'loss': ['hinge', 'log_loss', 'modified_huber'],
    'alpha': [0.0001, 0.001, 0.01, 0.1]
}

best_grid_score = -1
best_grid_params = None
grid_results = []

start_time = time.time()

# Генерация всех комбинаций параметров
for loss in param_grid['loss']:
    for alpha in param_grid['alpha']:
        # Вложенная функция для создания модели с текущими
        # гиперпараметрами.
        def create_model(l=loss, a=alpha):
            return SGDClassifier(
                loss=l,
                alpha=a,
                random_state=1
            )

        score = kfold_cv(create_model, accuracy_score, x_train, y_train)
        grid_results.append((score, loss, alpha))

        if score > best_grid_score:
            best_grid_score = score
            best_grid_params = {'loss': loss, 'alpha': alpha}

grid_time = time.time() - start_time

print(f"Лучшие параметры: {best_grid_params}")
print(f"Лучший CV Score: {best_grid_score:.4f}")
print(f"Время выполнения: {grid_time:.2f} сек")

Grid Search
Лучшие параметры: {'loss': 'hinge', 'alpha': 0.01}
Лучший CV Score: 0.6727
Время выполнения: 1.15 сек


In [116]:
# Используйте найденные гиперпараметры для обучения модели.
# Протестируйте модель на x_test, y_test.
# Сравните полученные результаты с теми, что получены в пункте 0.

best_grid_model = SGDClassifier(**best_grid_params, random_state=1)
best_grid_model.fit(x_train, y_train)
y_pred_grid = best_grid_model.predict(x_test)
grid_accuracy = accuracy_score(y_test, y_pred_grid)

print(f"\nТестовый Accuracy с Grid Search: {grid_accuracy:.4f}")
print(f"\nAccuracy базовая модель: {base_accuracy:.4f}")

print("\nМатрица ошибок с Grid Search:")
print(confusion_matrix(y_test, y_pred_grid))

print("\nМатрица ошибок базовая модель:")
print(confusion_matrix(y_test, y_pred_base))


Тестовый Accuracy с Grid Search: 0.7333

Accuracy базовая модель: 0.6000

Матрица ошибок с Grid Search:
[[3 0 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0]
 [0 0 0 2 0 0 0 1 0 0]
 [0 0 0 0 3 0 0 0 0 0]
 [0 0 0 0 0 3 0 0 0 0]
 [1 0 0 0 0 1 1 0 0 0]
 [0 0 0 0 0 0 0 3 0 0]
 [0 0 1 0 0 1 0 0 1 0]
 [0 0 0 0 0 0 0 1 0 2]]

Матрица ошибок базовая модель:
[[3 0 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0]
 [0 0 1 2 0 0 0 0 0 0]
 [0 0 0 0 2 1 0 0 0 0]
 [0 0 0 0 0 3 0 0 0 0]
 [1 0 0 0 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 3 0 0]
 [0 0 1 0 0 1 0 0 1 0]
 [0 0 1 0 0 0 0 2 0 0]]


## 3. Random search

In [117]:
# 1. Реализуйте алгоритм поиска гиперпараметров random search.
# 2. Запустите поиск гиперпараметров, замерьте время работы
# алгоритма.
# 3. Выведите найденные значения гиперпараметров и время работы.
# Замечание: x_test, y_test не должны применяться в рамках
# данного алгоритма.
# Замечание: убедитесь, что гиперпараметры по умолчанию включены
# в пространство поиска.
# Требование: используйте kfold_cv для получения значения метрики
# в рамках одной итерации поиска гиперпараметров.
# Требование: количество итераций должно быть меньше в сравнении
# с grid search.

print("Random Search")

param_dist = {
    'loss': ['hinge', 'log_loss', 'modified_huber'],
    'alpha': (0.0001, 0.1)  # непрерывный диапазон для alpha
}
n_iter = 5

best_random_score = -1
best_random_params = None
random_results = []

start_time = time.time()

for _ in range(n_iter):
    loss = np.random.choice(param_dist['loss'])
    # Для alpha используем np.random.uniform для выборки из
    # непрерывного диапазона.
    alpha = np.random.uniform(param_dist['alpha'][0],
                              param_dist['alpha'][1])

    # Вложенная функция для создания модели с текущими
    # гиперпараметрами.
    def create_model(l=loss, a=alpha):
        return SGDClassifier(
            loss=l,
            alpha=a,
            random_state=1
        )

    score = kfold_cv(create_model, accuracy_score, x_train, y_train)
    random_results.append((score, loss, alpha))

    if score > best_random_score:
        best_random_score = score
        best_random_params = {'loss': loss, 'alpha': alpha}

random_time = time.time() - start_time

print(f"Лучшие параметры: {best_random_params}")
print(f"Лучший CV Score: {best_random_score:.4f}")
print(f"Время выполнения: {random_time:.2f} сек")

Random Search
Лучшие параметры: {'loss': np.str_('hinge'), 'alpha': 0.01049960222608993}
Лучший CV Score: 0.6455
Время выполнения: 0.41 сек


In [118]:
# Используйте найденные гиперпараметры для обучения модели.
# Протестируйте модель на x_test, y_test (accuracy, матрица
# ошибок).
# Сравните полученные результаты с теми, что получены в пункте 0.
# Сравните полученные результаты с теми, что получены в пункте 2.

best_random_model = SGDClassifier(**best_random_params,
                                  random_state=1)
best_random_model.fit(x_train, y_train)
y_pred_random = best_random_model.predict(x_test)
random_accuracy = accuracy_score(y_test, y_pred_random)

print(f"\nБазовая модель Accuracy: {base_accuracy:.4f}")
print(f"\nGrid Search Accuracy: {grid_accuracy:.4f}")
print(f"\nRandom Search Accuracy: {random_accuracy:.4f}")

print("\nМатрица ошибок базовая модель:")
print(confusion_matrix(y_test, y_pred_base))

print("\nМатрица ошибок с Grid Search:")
print(confusion_matrix(y_test, y_pred_grid))

print("\nМатрица ошибок с Random Search:")
print(confusion_matrix(y_test, y_pred_random))


Базовая модель Accuracy: 0.6000

Grid Search Accuracy: 0.7333

Random Search Accuracy: 0.7000

Матрица ошибок базовая модель:
[[3 0 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0]
 [0 0 1 2 0 0 0 0 0 0]
 [0 0 0 0 2 1 0 0 0 0]
 [0 0 0 0 0 3 0 0 0 0]
 [1 0 0 0 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 3 0 0]
 [0 0 1 0 0 1 0 0 1 0]
 [0 0 1 0 0 0 0 2 0 0]]

Матрица ошибок с Grid Search:
[[3 0 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0]
 [0 0 0 2 0 0 0 1 0 0]
 [0 0 0 0 3 0 0 0 0 0]
 [0 0 0 0 0 3 0 0 0 0]
 [1 0 0 0 0 1 1 0 0 0]
 [0 0 0 0 0 0 0 3 0 0]
 [0 0 1 0 0 1 0 0 1 0]
 [0 0 0 0 0 0 0 1 0 2]]

Матрица ошибок с Random Search:
[[3 0 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 0 0 0 0]
 [1 0 1 0 0 0 0 0 1 0]
 [0 0 0 2 0 0 0 1 0 0]
 [0 0 0 0 3 0 0 0 0 0]
 [0 0 0 0 0 2 0 0 1 0]
 [1 0 0 0 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 3 0 0]
 [0 0 1 0 0 0 0 0 2 0]
 [0 0 0 0 0 0 0 1 0 2]]


## 4. Доп. задание (опционально)

### 4.1 Bayesian optimization

Примените байесовскую оптимизацию для поиска гиперпараметров.
В качестве алгоритма используйте `BayesSearchCV` из пакета `scikit-optimize`.

Сложность: почти бесплатный балл.

In [119]:
# 1. Инстанцируйте BayesSearchCV.
# 2. Запустите поиск гиперпараметров, замерьте время
# работы алгоритма.
# 3. Выведите найденные значения гиперпараметров
# и время работы.

print("Bayesian Optimization")

# Определение пространства поиска
bayes_space = {
    'loss': Categorical(['hinge', 'log_loss', 'modified_huber']),
    'alpha': Real(0.0001, 0.1, prior='log-uniform')
}

# Создание и обучение BayesSearchCV
start_time = time.time()
bayes_search = BayesSearchCV(
    estimator=SGDClassifier(random_state=1),
    search_spaces=bayes_space,
    n_iter=5,
    cv=5,
    random_state=1,
    n_jobs=1
)

bayes_search.fit(x_train, y_train)
bayes_time = time.time() - start_time

print(f"Лучшие параметры: {bayes_search.best_params_}")
print(f"Лучший CV Score: {bayes_search.best_score_:.4f}")
print(f"Время выполнения: {bayes_time:.2f} сек")


Bayesian Optimization
Лучшие параметры: OrderedDict({'alpha': 0.011237567661049233, 'loss': 'log_loss'})
Лучший CV Score: 0.6273
Время выполнения: 0.63 сек


In [120]:
# Используйте найденные гиперпараметры для обучения модели.
# Протестируйте модель на x_test, y_test (accuracy, матрица ошибок).
# Сравните полученные результаты с теми, что получены в пункте 0.
# Сравните полученные результаты с теми, что получены в пункте 2.

y_pred_bayes = bayes_search.predict(x_test)
bayes_accuracy = accuracy_score(y_test, y_pred_bayes)

print(f"\nBayesSearchCV Accuracy : {bayes_accuracy:.4f}")
print(f"\nБазовая модель Accuracy: {base_accuracy:.4f}")
print(f"\nGrid Search Accuracy: {grid_accuracy:.4f}")
print(f"\nRandom Search Accuracy: {random_accuracy:.4f}")

print("\nМатрица ошибок с BayesSearchCV:")
print(confusion_matrix(y_test, y_pred_bayes))

print("\nМатрица ошибок с базовой моделью:")
print(confusion_matrix(y_test, y_pred_base))

print("\nМатрица ошибок с Grid Search:")
print(confusion_matrix(y_test, y_pred_grid))

print("\nМатрица ошибок с Random Search:")
print(confusion_matrix(y_test, y_pred_random))



BayesSearchCV Accuracy : 0.6333

Базовая модель Accuracy: 0.6000

Grid Search Accuracy: 0.7333

Random Search Accuracy: 0.7000

Матрица ошибок с BayesSearchCV:
[[3 0 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0]
 [0 1 0 2 0 0 0 0 0 0]
 [0 0 0 0 3 0 0 0 0 0]
 [1 0 0 0 0 2 0 0 0 0]
 [2 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 3 0 0]
 [0 0 1 0 0 1 0 0 1 0]
 [0 0 0 0 0 1 0 1 0 1]]

Матрица ошибок с базовой моделью:
[[3 0 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0]
 [0 0 1 2 0 0 0 0 0 0]
 [0 0 0 0 2 1 0 0 0 0]
 [0 0 0 0 0 3 0 0 0 0]
 [1 0 0 0 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 3 0 0]
 [0 0 1 0 0 1 0 0 1 0]
 [0 0 1 0 0 0 0 2 0 0]]

Матрица ошибок с Grid Search:
[[3 0 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0]
 [0 0 0 2 0 0 0 1 0 0]
 [0 0 0 0 3 0 0 0 0 0]
 [0 0 0 0 0 3 0 0 0 0]
 [1 0 0 0 0 1 1 0 0 0]
 [0 0 0 0 0 0 0 3 0 0]
 [0 0 1 0 0 1 0 0 1 0]
 [0 0 0 0 0 0 0 1 0 2]]

Матрица ошибок с Random Search:
[[3 0 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 0 0 0 0]
 

### 4.2 Tree of Parzen Estimators (TPE) из HyperOpt

Примените TPE из библиотеки hyperopt для поиска гиперпараметров. Вики по HyperOpt: https://github.com/hyperopt/hyperopt/wiki/FMin

Сложность: чтец документаций o(*￣▽￣*)ブ.

In [121]:
def objective(args):
    """
    Принимает гиперпараметры, инстанцирует модель, обучает её,
    возвращает значение метрики.

    Замечание: x_test, y_test не должны применяться в рамках
    данного алгоритма.
    """
    model = SGDClassifier(
        loss=args['loss'],
        alpha=float(args['alpha']),
        random_state=1
    )

    score = kfold_cv(
        lambda: model, accuracy_score, x_train, y_train
    )
    return {'loss': 1 - score, 'status': STATUS_OK}


In [122]:
# Определите пространство поиска гиперпараметров
space = {
    'loss': hp.choice('loss', ['hinge', 'log_loss', 'modified_huber']),
    'alpha': hp.uniform('alpha', 0.0001, 0.1)  # Используем равномерное распределение
}

In [123]:
# 1. Запустите поиск гиперпараметров, замерьте время работы алгоритма.
# 2. Выведите найденные значения гиперпараметров и время работы.

# Запуск оптимизации
start_time = time.time()
trials = Trials()

best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=5,
    trials=trials
)
tpe_time = time.time() - start_time

loss_options = ['hinge', 'log_loss', 'modified_huber']
best_tpe_args = {
    'loss': loss_options[best['loss']],
    'alpha': best['alpha']
}

print(f"Лучшие параметры: {best_tpe_args}")
best_tpe_score = 1 - trials.best_trial['result']['loss']
print(f"Лучший CV Score: {best_tpe_score:.4f}")
print(f"Время выполнения: {tpe_time:.2f} сек")

100%|██████████| 5/5 [00:00<00:00, 10.13trial/s, best loss: 0.34545454545454546]
Лучшие параметры: {'loss': 'log_loss', 'alpha': np.float64(0.04565321940366501)}
Лучший CV Score: 0.6545
Время выполнения: 0.50 сек


In [124]:
# Используйте найденные гиперпараметры для обучения модели.
# Протестируйте модель на x_test, y_test (accuracy, матрица ошибок).
# Сравните полученные результаты с теми, что получены в пункте 0.
# Сравните полученные результаты с теми, что получены в пункте 2.

best_tpe_model = SGDClassifier(**best_tpe_args, random_state=1)
best_tpe_model.fit(x_train, y_train)
y_pred_tpe = best_tpe_model.predict(x_test)
tpe_accuracy = accuracy_score(y_test, y_pred_tpe)

print(f"\nTPE Accuracy: {tpe_accuracy:.4f}")
print(f"\nBayesSearchCV Accuracy : {bayes_accuracy:.4f}")
print(f"\nБазовая модель Accuracy: {base_accuracy:.4f}")
print(f"\nGrid Search Accuracy: {grid_accuracy:.4f}")
print(f"\nRandom Search Accuracy: {random_accuracy:.4f}")

print("\nМатрица ошибок TPE:")
print(confusion_matrix(y_test, y_pred_tpe))

print("\nМатрица ошибок с BayesSearchCV:")
print(confusion_matrix(y_test, y_pred_bayes))

print("\nМатрица ошибок с базовой моделью:")
print(confusion_matrix(y_test, y_pred_base))

print("\nМатрица ошибок с Grid Search:")
print(confusion_matrix(y_test, y_pred_grid))

print("\nМатрица ошибок с Random Search:")
print(confusion_matrix(y_test, y_pred_random))


TPE Accuracy: 0.7333

BayesSearchCV Accuracy : 0.6333

Базовая модель Accuracy: 0.6000

Grid Search Accuracy: 0.7333

Random Search Accuracy: 0.7000

Матрица ошибок TPE:
[[3 0 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0]
 [1 0 0 2 0 0 0 0 0 0]
 [0 0 0 0 3 0 0 0 0 0]
 [1 0 0 0 0 2 0 0 0 0]
 [1 0 0 0 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 3 0 0]
 [0 0 0 0 0 0 0 0 3 0]
 [0 0 0 0 0 0 0 1 0 2]]

Матрица ошибок с BayesSearchCV:
[[3 0 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0]
 [0 1 0 2 0 0 0 0 0 0]
 [0 0 0 0 3 0 0 0 0 0]
 [1 0 0 0 0 2 0 0 0 0]
 [2 0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 3 0 0]
 [0 0 1 0 0 1 0 0 1 0]
 [0 0 0 0 0 1 0 1 0 1]]

Матрица ошибок с базовой моделью:
[[3 0 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 0 0 0 0]
 [1 1 1 0 0 0 0 0 0 0]
 [0 0 1 2 0 0 0 0 0 0]
 [0 0 0 0 2 1 0 0 0 0]
 [0 0 0 0 0 3 0 0 0 0]
 [1 0 0 0 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 3 0 0]
 [0 0 1 0 0 1 0 0 1 0]
 [0 0 1 0 0 0 0 2 0 0]]

Матрица ошибок с Grid Search:
[[3 0 0 0 0 0 0 0 0 0]
 [0 3 0 0 0 0 